# Contact point and swing timing

This notebook reconstructs, for every tracked MLB swing, the three per-swing
numbers Baseball Savant publishes only in aggregate on its Swing Timing and Miss
Distance leaderboard, and validates them against Savant's own published
aggregates on two seasons nothing here was fit on.

| Axis | Savant field | Units | Sign | Category |
|---|---|---|---|---|
| x | `delta_batball_tiedup_neg_x2` | inches | tied up negative, flail positive | centered within +/- 4 in |
| y | `delta_batball_late_neg_y2_msec` | ms | late negative, early positive | on time within +/- 7 ms |
| z | `ball_pos_above_plane` | inches | over negative, under positive | lined up within +/- 2 in |

Perfect is contact and centered and on time and lined up. Flawed is a whiff that
is none of the three. Nothing here touches a database: every cell reads cached
files under `data/`. 2025 is the only season anything is fit on; 2024 and 2026
are holdouts.

Design: `docs/superpowers/specs/2026-09-13-contact-point-design.md`.
Plan: `docs/superpowers/plans/2026-09-13-contact-point-plan.md`.


In [1]:
import json, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image, display, Markdown

sys.path.insert(0, ".")
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

from cp_lib import join, geometry, discover, reconstruct, calibrate, validate, plots, traits
from cp_lib import savant_truth as st

REPORTS = Path("data/reports")
FIGURES = Path("data/figures")
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
FIT = 2025
SEASONS = tuple(int(p.stem.split("_")[1]) for p in sorted(Path("data").glob("swings_*.parquet"))
                if (Path("data/savant") / f"done_{p.stem.split('_')[1]}.json").exists())
snap = [json.loads(l) for l in open("data/pull_manifest.jsonl")]
SNAPSHOT = {int(r["season"]): r for r in snap}
LAST_GAME = {}
print("seasons with a complete Savant cache:", SEASONS)
for s in SEASONS:
    r = SNAPSHOT.get(s, {})
    LAST_GAME[s] = str(pd.read_parquet(f"data/swings_{s}.parquet", columns=["game_date"])
                       ["game_date"].max())
    print(f"  {s}: {r.get('rows', '?')} rows, pulled {r.get('finished_utc', '?')}, "
          f"last game {LAST_GAME[s]}")
HOLDOUT = tuple(s for s in SEASONS if s != FIT)
print("fit season:", FIT, " holdouts:", HOLDOUT)


seasons with a complete Savant cache: (2024, 2025, 2026)
  2024: 340278 rows, pulled 2026-09-13T05:03:59+00:00, last game 2024-09-30
  2025: 339082 rows, pulled 2026-09-13T05:03:27+00:00, last game 2025-09-28
  2026: 310405 rows, pulled 2026-09-13T05:04:38+00:00, last game 2026-09-11
fit season: 2025  holdouts: (2024, 2026)


## 0. Scorecard

In [2]:
# SCORECARD. Written by the last cell; this notebook is executed twice so it
# shows the final table here on the second pass.
p = REPORTS / "scorecard.csv"
if p.exists():
    sc = pd.read_csv(p)
    display(sc)
    hdr = REPORTS / "scorecard_header.json"
    if hdr.exists():
        for k, v in json.load(open(hdr)).items():
            print(f"{k}: {v}")
else:
    print("(the scorecard is written by the last cell; run the notebook twice)")


,season,table,in_sample,calibration_target,mean_mae_rates,mean_r_rates,players
0,2024,batter by contact type,False,False,0.0428,0.503,487
1,2024,batter rates,False,False,0.0369,0.662,512
2,2024,pitcher by contact type,False,False,0.0267,0.695,560
3,2024,pitcher rates,False,False,0.0203,0.876,598
4,2025,batter by contact type,True,False,0.0412,0.508,490
5,2025,batter rates,True,False,0.0359,0.677,523
6,2025,pitcher by contact type,True,True,0.0259,0.693,570
7,2025,pitcher rates,True,False,0.0205,0.861,618
8,2026,batter by contact type,False,False,0.0436,0.507,484
9,2026,batter rates,False,False,0.0369,0.663,512


swing filter: bat_speed_nobunt
swing filter median abs rel error: pitcher 0.0021, batter 0.0048
tail join rate: 0.99995 (21328 of 21329, 1 unmatched)
discovery: x=fallback(r2 0.712)  y=recovered(r2 0.951)  z=fallback(r2 0.665)
2026 snapshot: last game date: 2026-09-11
seasons: fit 2025, holdouts (2024, 2026)


## 1. Which of our rows is a swing

Savant's leaderboard counts a swing its own way. Before any number is compared,
our denominator has to match its `n_swings` per player. Three nested rules are
scored against both the pitcher and the batter CSV; the winner is the one with
the lowest median absolute relative error averaged over the two.


In [3]:
df25 = join.load_swings(FIT)
rep = pd.concat([join.denominator_report(df25, FIT, t) for t in ("pitcher", "batter")],
                ignore_index=True)
display(rep)
RULE = join.choose_swing_filter(df25, FIT)
print("chosen swing filter:", RULE)
rep.to_csv(REPORTS / "denominator_2025.csv", index=False)

# spec section 3: what the chosen rule still gets wrong, player by player.
res = [join.residual_report(df25, FIT, t, RULE) for t in ("pitcher", "batter")]
print("\nresidual mismatch under", RULE, "(absolute, per player):")
for r in res:
    print(f"  {r['type']:8s} players={r['players']:4d} ours={r['total_ours']} "
          f"savant={r['total_savant']} median={r['median_diff']:.1f} "
          f"p05={r['p05']:.1f} p95={r['p95']:.1f} exact={r['exact_match']:.3f}")

nulls = {}
sw25 = df25[join.swing_filter(df25, RULE)]
for c in ("intercept_ball_minus_batter_pos_x_inches", "intercept_ball_minus_batter_pos_y_inches",
          "attack_direction", "attack_angle", "swing_path_tilt", "bat_speed",
          "launch_speed", "launch_angle"):
    nulls[c] = f"{int(sw25[c].isna().sum())} ({sw25[c].isna().mean()*100:.2f}%)"
print("\nnulls on the", len(sw25), "swings the rule keeps:")
for k, v in nulls.items():
    print(f"  {k:46s} {v}")

# spec section 3: the histogram endpoint's total against the CSV n_swings
h = st.load_histograms(FIT, "batter")
lb = st.load_leaderboard(FIT, "batter")[["id", "n_swings"]].dropna(subset=["id"]).astype({"id": int})
tot = h.groupby(["id", "axis"])["n"].sum().unstack()
m = lb.merge(tot.reset_index(), on="id", how="inner")
print(f"\nhistogram total minus CSV n_swings, {len(m)} batters:")
for ax in ("x", "y", "z"):
    d = m[ax] - m["n_swings"]
    print(f"  axis {ax}: sum {int(d.sum()):6d}  exact match {float((d == 0).mean()):.3f}")
print("  The y histogram matches n_swings for every batter. x and z run about")
print("  0.15 percent high, the same rate at which our own intercept columns are")
print("  null, so Savant has an x and a z on swings whose timing value does not")
print("  exist. That is what an intercept-depth definition of y predicts.")


,rule,type,players,median_abs_rel_err,p90_abs_rel_err,total_ours,total_savant
0,bat_speed,pitcher,826,0.006525,0.018224,329848,326997
1,bat_speed_nobunt,pitcher,826,0.002119,0.009927,328468,326997
2,bat_speed_nobunt_typical,pitcher,826,0.014635,0.031782,322318,326997
3,bat_speed,batter,671,0.007795,0.035714,330418,326997
4,bat_speed_nobunt,batter,671,0.004756,0.018868,329039,326997
5,bat_speed_nobunt_typical,batter,671,0.011885,0.027778,322890,326997


chosen swing filter: bat_speed_nobunt



residual mismatch under bat_speed_nobunt (absolute, per player):
  pitcher  players= 826 ours=328468 savant=326997 median=1.0 p05=0.0 p95=6.0 exact=0.410
  batter   players= 671 ours=329039 savant=326997 median=2.0 p05=0.0 p95=9.0 exact=0.255

nulls on the 329040 swings the rule keeps:
  intercept_ball_minus_batter_pos_x_inches       487 (0.15%)
  intercept_ball_minus_batter_pos_y_inches       487 (0.15%)
  attack_direction                               0 (0.00%)
  attack_angle                                   0 (0.00%)
  swing_path_tilt                                2 (0.00%)
  bat_speed                                      0 (0.00%)
  launch_speed                                   100509 (30.55%)
  launch_angle                                   100127 (30.43%)



histogram total minus CSV n_swings, 671 batters:
  axis x: sum    481  exact match 0.641
  axis y: sum      0  exact match 1.000
  axis z: sum    481  exact match 0.641
  The y histogram matches n_swings for every batter. x and z run about
  0.15 percent high, the same rate at which our own intercept columns are
  null, so Savant has an x and a z on swings whose timing value does not
  exist. That is what an intercept-depth definition of y predicts.


## 2. What Savant's three numbers actually are

Savant's own tail rows rule out the obvious geometry. They carry x values past
100 inches on a 34 inch bat with miss distances of 10 inches, so x is not an
along-bat offset from the sweet spot at closest approach. So the first modeling
phase is discovery: fit a ladder of candidates to the labeled whiff tails and
see which, if any, reproduces Savant's value and its category label.

The tails are the ten worst whiffs per list per player-season, selected on the
label itself. They identify functional form, sign and scale. Nothing that ships
is centered or scaled on them.


In [4]:
disc = json.load(open(REPORTS / "discovery_2025.json"))
print("labeled rows:", disc["n_labeled"], " join:", disc["join"])
print("join rate: %.5f  (%d of %d tails matched, %d unmatched)   y_com_ft: %s   tilt_sign: %s"
      % (disc["join_rate"], disc["join"]["exact"] + disc["join"]["nearest"],
         disc["join"]["tails"], disc["join"]["unmatched"], disc["y_com_ft"], disc["tilt_sign"]))
ladder = pd.DataFrame(disc["ladder"])
display(ladder)
for ax in ("x", "y", "z"):
    v = disc["verdicts"][ax]
    b = v["best"]
    print(f"axis {ax}: {v['status'].upper():9s} best {b['name']:22s} "
          f"r2 {b['r2']:.3f}  label agreement {b['label_agreement']:.3f}  "
          f"all_swing {b['all_swing']}")
display(Markdown(open(REPORTS / "discovery_2025.md").read()))


labeled rows: 21328  join: {'tails': 21329, 'exact': 21328, 'nearest': 0, 'unmatched': 1, 'fanout_rows_dropped': 12}
join rate: 0.99995  (21328 of 21329 tails matched, 1 unmatched)   y_com_ft: 0.0   tilt_sign: 1.0


,axis,candidate,cost,n_features,n,r2,label_agreement,slope,all_swing,chosen,status
0,x,icpt_x_body_by_stand,1,1,21328,0.298,0.724,1.000,True,False,fallback
1,x,icpt_x_body,1,1,21328,0.296,0.724,0.614,True,False,fallback
2,x,icpt_x_field,1,1,21328,0.022,0.724,0.041,True,False,fallback
3,x,icpt_xy_body,2,2,21328,0.457,0.770,1.000,True,False,fallback
4,x,ball_lateral_at_icpt,2,3,21328,0.306,0.729,1.000,True,False,fallback
5,x,along_bat_from_com,3,1,21328,0.060,0.716,1.000,True,False,fallback
6,x,along_bat_plus_icpt,3,4,21328,0.690,0.866,1.000,True,False,fallback
7,x,x_full_geometry,4,9,21328,0.712,0.864,1.000,True,True,fallback
8,y,icpt_y_over_ballspeed,1,1,21328,0.951,0.914,1.000,True,True,recovered
9,y,icpt_y_by_stand,1,1,21328,0.950,0.916,1.000,True,False,recovered


axis x: FALLBACK  best x_full_geometry        r2 0.712  label agreement 0.864  all_swing True
axis y: RECOVERED best icpt_y_over_ballspeed  r2 0.951  label agreement 0.914  all_swing True
axis z: FALLBACK  best z_miss_decomp          r2 0.665  label agreement 0.708  all_swing False


# What Savant's three per-swing numbers are

Discovery on the 2025 labeled whiff tails. Written by Task 4 of the plan.
Every number here is measured; the machine-readable version is
`discovery_2025.json` and the full ladder is its `ladder` key.

## The labeled set

Savant's per-swing tail endpoint returns, for each player-season, the ten worst
whiffs by each of four quantities (miss distance, x, y, z), under both the
pitcher and the batter view. Pooled over the 1,497 players in the 2025
leaderboards and deduplicated by `play_id`, that is 21,329 distinct swings.
7,587 of them were selected by exactly one list and 152 by all eight.

Joined to our own rows on (game_pk, batter, pitcher, miss distance rounded to
six places): **21,328 matched exactly, 1 unmatched, join rate 0.99995**. No row
needed the nearest-distance fallback. Twelve extra rows came out of the exact
merge because two of our swings shared a join key; those are deduplicated to one
row per `play_id` and counted rather than dropped silently.

All 21,328 are whiffs, which is what the endpoint's leaked SQL says they should
be. The one row that read as a foul before turned out to be a foul tip, which
Savant counts as a whiff: our whiff rate is 0.2346 with foul tips called contact
and 0.2565 with them called whiffs, against Savant's published 0.2569.

Free parameters: `y_com_ft = 0.0` and `tilt_sign = +1`. See the bottom of this
file for how each was chosen.

## y is the contact depth expressed as the ball's flight time. RECOVERED.

**r-squared 0.951, label agreement 0.914, slope 1.00, all-swing.**

The winning candidate is the cheapest one on the ladder:

    sav_y = 0.924 * (intercept depth / ball speed at that depth) - 17.92    (RHB)
    sav_y = 0.911 * (intercept depth / ball speed at that depth) - 18.34    (LHB)

with depth in inches, ball speed in inches per millisecond, and the result in
milliseconds. The raw depth in inches gives almost the same fit and its pooled
slope is 0.699 ms per inch, whose reciprocal is 1.43 inches per millisecond, or
81.3 mph. That is the ball's speed near the plate. So Savant's early/late number
is not a bat-relative geometry at all. It is how much earlier or later than an
ideal depth the bat met the ball, converted to time at the speed the ball was
travelling.

The coefficient on the time form is 0.92, not 1.00, and the intercept puts the
zero crossing at 19.4 ms of ball flight for right-handed batters and 20.1 ms for
left-handed ones, which at the tails' median ball speed of 1.40 inches per
millisecond (79.5 mph) is **27.2 and 28.2 inches of contact depth**. Savant's "on time" is contact about 27 inches in
front of the batter's own reference point, and the +/- 7 ms band is about +/- 10
inches of depth around it.

Two consequences worth recording. First, the sign convention falls out: deeper
contact is later, so positive is early, which is Savant's convention.
Second, Savant's own published histograms corroborate the definition. Over the
2025 batters, the y histogram's total equals the leaderboard's `n_swings` for
every single player, while the x and z histograms run about 0.15 percent high,
which is the exact rate at which our own intercept columns are null. Savant has
an x and a z on swings whose timing value does not exist, which is what a
definition built on the intercept depth predicts and what a definition built on
anything else does not.

Every other y candidate is a proxy for the same thing. `attack_direction`
reaches r-squared 0.902 on its own and `swing_length` correlates 0.89, because
on a whiff all three move together with how far out front the bat was.

## x is not recovered. MODELED.

**Best candidate r-squared 0.712, label agreement 0.864. Both gates missed.**

The ladder's best is `x_full_geometry`, a nine-feature per-stand linear model
over the intercept location, attack direction, bat speed, the ball's position at
the intercept depth and the strike-zone-relative height. The cheapest candidate
that comes close is `along_bat_plus_icpt` at 0.690 with four features.

What x is not: on the tails it runs from -163.5 to +143.9 inches while the miss
distance on those same swings never exceeds 57.5 inches, and only 16.9 percent
of rows satisfy |x| <= miss distance. A signed offset along a 34 inch bat at the
instant of closest approach cannot do that, and neither can any component of the
3D miss vector. Whatever x measures, it is not measured at the instant miss
distance is measured.

The single strongest public quantity is the lateral intercept distance
(r = 0.544 pooled, and the same sign for both stands because that column is
already body-relative), followed by miss distance itself at 0.452 and the
intercept depth at 0.375. The intercept depth mattering at all is the tell: a
purely lateral, purely at-contact quantity should not care how far out front the
bat was.

Task 4 Step 5's first expensive hypothesis, the bat-frame projection taken at
the instant the ball crosses the swing plane, **is not computable from the
published fields**. It needs the bat's position in three dimensions, and
Statcast publishes the intercept point's lateral offset and depth but no height,
so the plane cannot be anchored. Anchoring it at the ball's own position at the
intercept depth makes the crossing instant the intercept instant by
construction, which is degenerate. This is recorded as unreachable rather than
tried and failed.

## z is not recovered. MODELED.

**Best candidate r-squared 0.665, label agreement 0.708. Both gates missed.**

The ladder's best is `z_miss_decomp`, which is Task 4 Step 5's third hypothesis
(z as a component of the miss vector, `sign(z_above_plane) * min(|z_above_plane|,
miss_distance)`) plus the strike-zone-relative ball height, the pitch descent
angle and the miss distance. The hypothesis itself is refuted as a definition:
only 53.6 percent of tail rows satisfy |z| <= miss distance, so z is not a
component of the miss vector either. It earns its place as a feature, not as an
explanation.

z does behave like contact-instant geometry, which is what the spec expected.
The ball's height at the intercept depth correlates 0.724 with it and the
pitch's descent angle 0.700, both with the same sign for both stands: a higher,
flatter pitch means the swing passed under it, and Savant's positive z is
"under". The strike-zone-relative height correlates identically at 0.724.

What is missing is the bat's own height at contact, which Statcast does not
publish. Everything the ladder can reach is the ball's side of the difference,
so the fits saturate near 0.6.

The plan's own cheap z candidate, the ball's displacement out of the swing plane
between the intercept depth and the plate, is worthless: r = -0.128, r-squared
0.016. It is recorded here because the plan named it first.

## The two free parameters

**tilt_sign = +1 (the barrel below the hands).** Geometry did not settle it: see
`attack_direction_sign.txt` for three measurements that disagree. The data did.
Over a 9 by 2 grid of (batter reference depth, tilt sign), the x ladder's best
r-squared is 0.695 at +1 against 0.611 at -1, while z prefers -1 by 0.0030. The
x preference is 28 times larger and agrees with how MLB depicts swing path tilt.

**y_com_ft = 0.0 (the point of the plate).** x and y are exactly insensitive to
it, as the plan predicted. z varies from 0.5619 to 0.5506 across the whole grid,
monotonically, with its maximum at the grid edge, which means the criterion does
not identify the parameter rather than that -2.0 feet is right. z falls back
either way, so the choice cannot change a verdict. 0.0 is the interpretable
anchor and is what ships.

## What this means downstream

y applies to every swing directly, from fields present on all of them. x and z
are modeled: a LightGBM regressor fit on these tails with inverse-selection
weights, applied to whiffs only, and the Driveline outcome inversion for contact
swings. The tails are the ten worst per list per player-season, selected on the
label, so they are whiff-only evidence and are treated as whiff-only evidence.

One alternative was identified and deliberately not pursued, because the plan
bounds the search here and the verdict is final once written. For balls in play
the bat and the ball met, so z is near zero by construction; a model of the
bat's height at the intercept could be learned from in-play swings and
subtracted from the ball's height to give z on whiffs. That is a different
experiment from the one this plan specifies, and it is the first thing to try if
anyone reopens z.


## 3. Reconstruct every swing, then calibrate

A recovered all-swing formula applies everywhere. An axis discovery did not
recover is treated as whiff-only evidence, which is what it is: its model is fit
on whiff tails and has never seen a swing where the bat met the ball. So whiffs
get the model and contact swings get the Driveline inversion, an exit-velocity
deficit against `1.23 * bat_speed + 0.23 * pitch_speed` for x and launch angle
minus attack angle for z. A contact swing with no launch data has no outcome to
invert; it stays unknown rather than being called centered.

Calibration is one monotone quantile map per axis onto Savant's published
per-player histograms, fit on 2025 and applied unchanged to every season.


In [5]:
verdicts, Y_COM, TILT = reconstruct.load_verdicts()
FALLBACK_AXES = [ax for ax, s in verdicts.items() if s.status == "fallback"]
print("fallback axes:", FALLBACK_AXES, " y_com_ft:", Y_COM, " tilt_sign:", TILT)

DF, PRED, CAL = {}, {}, {}
for s in SEASONS:
    d = join.load_swings(s)
    d = d[join.swing_filter(d, RULE)].reset_index(drop=True)
    DF[s] = d
    PRED[s] = reconstruct.reconstruct(d, s, verdicts, Y_COM, tilt_sign=TILT)
    print(f"season {s}: {len(d)} swings   sources "
          + "  ".join(f"{ax}:{dict(PRED[s]['source_' + ax].value_counts())}" for ax in ("x", "y", "z")))

# Fallback C: LightGBM on the 2025 tails with inverse-selection weights, whiffs only.
# Task 14 widened what it sees: the re-pull's per-pitch columns, per-batter traits
# computed on each season's OWN swings, MLB biography, and Savant's swing-path
# slate. Nothing here is a Savant per-swing label.
PLAYERS = traits.load_players()
SPATH = traits.swing_path()
IMPORTANCE = {}
if FALLBACK_AXES:
    tf = traits.tails_features(FIT, Y_COM, TILT, players=PLAYERS, sp=SPATH)
    hist_b = st.load_histograms(FIT, "batter")
    base = [c for c in geometry.ALL_SWING_FEATURE_COLUMNS if c in tf and tf[c].notna().any()]
    COLS = {ax: traits.fallback_columns(tf, base, axis=ax) for ax in FALLBACK_AXES}
    for ax, cc in COLS.items():
        print(f"\nfallback features, axis {ax}: {len(cc)} "
              f"({len(base)} geometry, {len(cc) - len(base)} new)")
    print("z keeps the geometry set: the new features help x on both holdouts and")
    print("hurt z on both, and traits.WIDENED_AXES carries the numbers.")
    for ax in FALLBACK_AXES:
        cols = COLS[ax]
        model = calibrate.fallback_fit(ax, tf, hist_b, cols, key="batter")
        IMPORTANCE[ax] = (pd.DataFrame({"feature": cols,
                                        "gain": model.feature_importance("gain")})
                          .sort_values("gain", ascending=False).head(10).reset_index(drop=True))
        for s in SEASONS:
            f0 = geometry.candidate_features(DF[s], s, y_com_ft=Y_COM, tilt_sign=TILT)
            f0["icpt_y_over_ballspeed"] = f0["icpt_y"] / f0["ball_in_per_ms"].replace(0, np.nan)
            feats = traits.extended_features(DF[s], f0, s, players=PLAYERS, sp=SPATH)
            sel = (PRED[s][f"source_{ax}"] == "fallback").to_numpy()
            if sel.any():
                PRED[s].loc[sel, f"{ax}_hat"] = calibrate.fallback_predict(
                    model, feats[sel], cols)
                PRED[s].loc[sel, f"source_{ax}"] = "model"
            del feats, f0
        print(f"  axis {ax}: filled the whiff rows in every season")
    for ax, t_ in IMPORTANCE.items():
        print(f"\ntop gain, axis {ax}")
        display(t_)

hist_p = st.load_histograms(FIT, "pitcher")
RATES = calibrate.split_rates(FIT, "pitcher")
MAPS_POOLED = calibrate.fit_calibration(PRED[FIT], DF[FIT], FIT, "pitcher", hist=hist_p,
                                        per_contact_type=False)
MAPS = calibrate.fit_calibration(PRED[FIT], DF[FIT], FIT, "pitcher", hist=hist_p, rates=RATES)
for s in SEASONS:
    CAL[s] = calibrate.apply_calibration(PRED[s], DF[s], MAPS)

# The calibration decision, shown rather than asserted. Savant publishes no
# per-contact-type histogram, so an earlier version fit each contact type onto the
# all-swing marginal, which forced every type to the all-swing rate. The split CSV
# does publish each type's category rates, and per_type_target rescales the pooled
# shape's three band masses onto them.
sav_ct = validate.savant_rates(FIT, "pitcher", "bat_contact_code")
POOLED_CAL = {s: calibrate.apply_calibration(PRED[s], DF[s], MAPS_POOLED) for s in SEASONS}
CELLS = [("in_play", "lined_up_percent"), ("whiff", "lined_up_percent"),
         ("in_play", "centered_percent"), ("foul", "centered_percent"),
         ("whiff", "centered_percent"), ("in_play", "on_time_percent"),
         ("foul", "on_time_percent"), ("whiff", "on_time_percent")]
AX_OF = {"centered_percent": "x", "on_time_percent": "y", "lined_up_percent": "z"}
rows = []
for ct, rate in CELLS:
    ax = AX_OF[rate]
    sel = (DF[FIT]["contact_type"] == ct).to_numpy()
    q = sav_ct[sav_ct.contact_type == ct]
    r = {"contact type": ct, "rate": rate,
         "savant": round(float(np.average(q[rate], weights=q.n_swings)), 4)}
    for name, vals in (("raw", PRED[FIT][f"{ax}_hat"]), ("pooled map", POOLED_CAL[FIT][f"{ax}_cal"]),
                       ("per-type map", CAL[FIT][f"{ax}_cal"])):
        lab = discover.label_of(ax, np.asarray(vals, float))
        mid = discover.LABELS[ax][1]
        r[name] = round(float(np.mean(lab[sel] == mid)), 4)
    r["known"] = round(float(np.isfinite(CAL[FIT].loc[sel, f"{ax}_cal"]).mean()), 4)
    rows.append(r)
print("\nThe league category rate per contact type, three calibration choices:")
display(pd.DataFrame(rows))
print("The per-type map lands on Savant wherever we can compute the axis at all.")
print("What is left is the 'known' column: a swing with no exit velocity has no x")
print("and no z, so it is Unknown rather than centered. The foul centered rate is")
print("0.726 times its 0.875 known share, which is the 0.635 in the table, and the")
print("in-play lined-up rate is 0.975 times 0.997. The residual is the missing")
print("swings, not the calibration.")

# Whether y should be per-type at all: it is the one axis with a recovered formula,
# so the pooled map might already be right. Measured on the HOLDOUTS, not on 2025.
MAPS_Y_POOLED = calibrate.fit_calibration(PRED[FIT], DF[FIT], FIT, "pitcher", hist=hist_p,
                                          rates=RATES, pooled_axes=("y",))
print("\ny per-type or y pooled, mean absolute error on the pitcher category rates:")
for name, mp in (("y per-type", MAPS), ("y pooled", MAPS_Y_POOLED)):
    line = []
    for s in SEASONS:
        c2 = calibrate.apply_calibration(PRED[s], DF[s], mp)
        sc = validate.compare_rates(validate.player_rates(DF[s], validate.categorize(c2, DF[s]),
                                                          "pitcher", c2),
                                    validate.savant_rates(s, "pitcher"))
        line.append(f"{s} {sc[sc.rate.isin(validate.RATE_COLS)]['mae'].mean():.4f}")
    print(f"  {name:11s} " + "   ".join(line))
print("  y stays per-type: it is better on both holdouts, not only on the fit season.")

print("\nper-player mean Wasserstein-1 against Savant's 2025 pitcher histograms.")
print("This is the price of the per-type map and it is a real regression: the")
print("target is no longer the pooled histogram those distances are measured on.")
for ax in ("x", "y", "z"):
    raw = calibrate.wasserstein_per_player(PRED[FIT][f"{ax}_hat"].to_numpy(float),
                                           DF[FIT], hist_p, ax, "pitcher")
    pol = calibrate.wasserstein_per_player(POOLED_CAL[FIT][f"{ax}_cal"].to_numpy(float),
                                           DF[FIT], hist_p, ax, "pitcher")
    per = calibrate.wasserstein_per_player(CAL[FIT][f"{ax}_cal"].to_numpy(float),
                                           DF[FIT], hist_p, ax, "pitcher")
    print(f"  axis {ax}: raw {raw.mean():7.3f}   pooled map {pol.mean():7.3f}   "
          f"per-type map {per.mean():7.3f}   players {len(per)}")


fallback axes: ['x', 'z']  y_com_ft: 0.0  tilt_sign: 1.0


season 2024: 315714 swings   sources x:{'inversion': np.int64(221447), 'fallback': np.int64(80165), 'geometry': np.int64(14089), 'no_outcome': np.int64(13)}  y:{'formula': np.int64(315714)}  z:{'inversion': np.int64(221734), 'fallback': np.int64(80165), 'no_outcome': np.int64(13815)}


season 2025: 329040 swings   sources x:{'inversion': np.int64(228531), 'fallback': np.int64(84413), 'geometry': np.int64(16085), 'no_outcome': np.int64(11)}  y:{'formula': np.int64(329040)}  z:{'inversion': np.int64(228913), 'fallback': np.int64(84413), 'no_outcome': np.int64(15714)}


season 2026: 300741 swings   sources x:{'inversion': np.int64(209499), 'fallback': np.int64(76399), 'geometry': np.int64(14827), 'no_outcome': np.int64(16)}  y:{'formula': np.int64(300741)}  z:{'inversion': np.int64(209817), 'fallback': np.int64(76399), 'no_outcome': np.int64(14525)}



fallback features, axis x: 80 (26 geometry, 54 new)

fallback features, axis z: 26 (26 geometry, 0 new)
z keeps the geometry set: the new features help x on both holdouts and
hurt z on both, and traits.WIDENED_AXES carries the numbers.


...

  y per-type  2024 0.0203   2025 0.0205   2026 0.0209


  y pooled    2024 0.0203   2025 0.0206   2026 0.0212
  y stays per-type: it is better on both holdouts, not only on the fit season.

per-player mean Wasserstein-1 against Savant's 2025 pitcher histograms.
This is the price of the per-type map and it is a real regression: the
target is no longer the pooled histogram those distances are measured on.


  axis x: raw   3.401   pooled map   0.598   per-type map   0.555   players 768


  axis y: raw   0.879   pooled map   0.728   per-type map   0.784   players 768


  axis z: raw   0.442   pooled map   0.208   per-type map   0.229   players 767


## 4. Validation

In the spec's fixed order, holdout seasons first and 2025 flagged in-sample.
Every rate is a fraction, not a percent, because that is how Savant's CSV
carries them.


In [6]:
results = {}
CATS = {s: validate.categorize(CAL[s], DF[s]) for s in SEASONS}
order = [s for s in SEASONS if s != FIT] + ([FIT] if FIT in SEASONS else [])

for kind, key in (("pitcher", "pitcher"), ("batter", "batter")):
    for s in order:
        ours = validate.player_rates(DF[s], CATS[s], key, CAL[s])
        cmp_ = validate.compare_rates(ours, validate.savant_rates(s, kind))
        results[(s, f"{kind} rates")] = cmp_
        print(f"\n=== {kind} rates, season {s}" + ("  (IN SAMPLE)" if s == FIT else "  (holdout)"))
        display(cmp_)

for kind, key in (("pitcher", "pitcher"), ("batter", "batter")):
    for s in order:
        t = validate.contact_type_table(DF[s], CATS[s], CAL[s], s, kind, key)
        results[(s, f"{kind} by contact type")] = t
        print(f"\n=== {kind} rates by contact type, season {s}"
              + ("  (IN SAMPLE)" if s == FIT else "  (holdout)"))
        display(t[t.rate.isin(["centered_percent", "on_time_percent", "lined_up_percent",
                               "perfect_percent", "flawed_percent"])])

# Perfect and flawed on their own. They are JOINT categories, perfect being a ball
# in play that is centered and on time and lined up, flawed a whiff that is none of
# the three. The per-type map sets each axis's marginal rate, so it has no direct
# hold on a joint one: agreement here is a check on whether the three axes are
# wrong together on the same swings, which is what a marginal map cannot fix.
print("\n=== perfect and flawed, the two joint categories, pitcher")
jr = []
for s in order:
    t = results[(s, "pitcher by contact type")]
    for ct, rate in (("in_play", "perfect_percent"), ("whiff", "flawed_percent")):
        r = t[(t.contact_type == ct) & (t.rate == rate)]
        if len(r):
            jr.append({"season": s, "cell": f"{ct} {rate}",
                       "ours": round(float(r.league_ours.iloc[0]), 4),
                       "savant": round(float(r.league_savant.iloc[0]), 4),
                       "mae": round(float(r["mae"].iloc[0]), 4),
                       "pearson_r": round(float(r["pearson_r"].iloc[0]), 3)})
display(pd.DataFrame(jr))

print("\n=== bucket means (the six avg_* values), pitcher")
for s in order:
    t = results[(s, "pitcher rates")]
    print(f"season {s}")
    display(t[t.rate.isin(validate.MEAN_COLS)])

# Attenuation. The per-player scatters have a slope well below 1: a pitcher Savant
# has near 0.75 lined up comes out near 0.68, one near 0.45 comes out near 0.48.
# That is what per-swing error does. The question is whether to undo it.
print("\n=== per-player attenuation, 2025, and what undoing it would cost")
rows = []
for kind in ("pitcher", "batter"):
    ours = validate.player_rates(DF[FIT], CATS[FIT], kind, CAL[FIT])
    sav = validate.savant_rates(FIT, kind)
    for ax in ("z", "x", "y"):
        col = calibrate.MID_RATE[ax]
        m = ours.merge(sav, on="id", suffixes=("_o", "_s"))
        m = m[m["n_swings_s"] >= 100].dropna(subset=[col + "_o", col + "_s"])
        a = m[col + "_s"].to_numpy(float)
        b = m[col + "_o"].to_numpy(float)
        sl = float(np.cov(a, b, ddof=1)[0, 1] / np.var(a, ddof=1))
        ideal = b.mean() + (b - b.mean()) / sl
        rows.append({"view": kind, "rate": col, "slope": round(sl, 3),
                     "pearson_r": round(float(np.corrcoef(a, b)[0, 1]), 3),
                     "players": len(m), "mae": round(float(np.abs(b - a).mean()), 4),
                     "mae_if_slope_1": round(float(np.abs(ideal - a).mean()), 4)})
display(pd.DataFrame(rows))
print("The last column is the answer. Stretching the rates until the slope is 1")
print("makes the error WORSE on all six, because the correlation is 0.66 to 0.92")
print("and not 1. With an imperfect predictor the least-error answer IS the shrunk")
print("one; a slope below 1 is what a noisy estimate should do, not a defect in it.")
print("Raising the slope and the accuracy together needs a better per-swing z, not")
print("a rescale of the rates it produces.")
K_MEASURED = calibrate.fit_reliability(CAL[FIT], DF[FIT], FIT, "pitcher")
print("\nThe per-swing version was measured too. Scaling each swing about its own")
print("player's season mean by k, the k that puts the pitcher slope at 1 is")
print("  " + "  ".join(f"{a} {v:.2f}" for a, v in sorted(K_MEASURED.items())))
print("and it is NOT applied. It moves every player's rate level far more than it")
print("moves the spread between players: pitcher category error goes from 0.021 to")
print("0.079 on the 2024 holdout, whiff centered from 0.404 to 0.699 against")
print("Savant's 0.402, and the pooled x histogram shifts by up to 0.186 in a single")
print("bin. calibrate.apply_reliability exists and is tested; nothing calls it.")

print("\n=== per-player Wasserstein-1 against Savant's histograms")
w_rows = []
for s in order:
    for kind, key in (("pitcher", "pitcher"), ("batter", "batter")):
        hh = st.load_histograms(s, kind)
        if hh.empty:
            continue
        for ax in ("x", "y", "z"):
            w = calibrate.wasserstein_per_player(CAL[s][f"{ax}_cal"].to_numpy(float),
                                                 DF[s], hh, ax, key)
            w_rows.append({"season": s, "type": kind, "axis": ax,
                           "mean_w1": round(float(w.mean()), 3),
                           "median_w1": round(float(w.median()), 3), "players": len(w)})
display(pd.DataFrame(w_rows))

print("\n=== league bins against TIMING_BINNED_DATA_LEAGUE, 2025")
BINS_TABLE = validate.league_bins_table(CAL[FIT], DF[FIT], FIT)
display(BINS_TABLE[BINS_TABLE.n_sav.notna() & (BINS_TABLE.n_sav > 200)].head(40))

# The x panel used to carry a spike Savant does not have: the Driveline loss curve
# stops at a 40 mph exit velocity deficit, which is 14 inches from the sweet spot,
# and np.interp clamped there, so every weakly hit ball past it landed on one
# value. The curve now continues at the slope of its final segment. That is not a
# claim about a bat 20 inches off the sweet spot; the quantile map sets the scale
# and what the continuation buys is rank order, so two different deficits keep two
# different values and a monotone map can spread them.
_x = PRED[FIT]["x_hat"]
_inv = (PRED[FIT]["source_x"] == "inversion").to_numpy()
_past = ((_x.abs() > 14.0) & _inv).sum()
_pile = _x[_inv].round(3).value_counts()
print(f"\ninversion rows past the curve's last published point: {int(_past)} "
      f"({_past / _inv.sum() * 100:.2f}% of them). Largest single x value shared by "
      f"more than one swing: {int(_pile.max())} swings at {float(_pile.idxmax()):+.3f} "
      f"inches, against 12,221 stacked on exactly 14 inches before the change.")



=== pitcher rates, season 2024  (holdout)


,rate,players,mae,pearson_r,league_ours,league_savant
0,tied_up_percent,598,0.014399,0.761774,0.081680,0.081000
1,centered_percent,598,0.026235,0.799434,0.653098,0.654636
2,flailed_percent,598,0.022617,0.863497,0.265179,0.264364
3,early_percent,598,0.018056,0.959972,0.202598,0.207316
4,on_time_percent,598,0.021068,0.909399,0.645043,0.642410
5,late_percent,598,0.020392,0.883091,0.151500,0.150274
6,over_percent,598,0.015939,0.923529,0.130310,0.131623
7,lined_up_percent,598,0.030779,0.846749,0.602967,0.622848
8,under_percent,598,0.029473,0.925841,0.222877,0.245529
9,perfect_percent,598,0.012668,0.911226,0.228612,0.224567



=== pitcher rates, season 2026  (holdout)


,rate,players,mae,pearson_r,league_ours,league_savant
0,tied_up_percent,580,0.014936,0.747095,0.090319,0.089212
1,centered_percent,580,0.028976,0.757325,0.646992,0.657885
2,flailed_percent,580,0.024053,0.858989,0.262634,0.252903
3,early_percent,580,0.016956,0.963866,0.202727,0.206033
4,on_time_percent,580,0.021567,0.909506,0.646743,0.645025
5,late_percent,580,0.020685,0.881144,0.149402,0.148941
6,over_percent,580,0.021075,0.901565,0.121488,0.136349
7,lined_up_percent,580,0.031964,0.845626,0.606995,0.628969
8,under_percent,580,0.024321,0.913261,0.222980,0.234682
9,perfect_percent,580,0.012573,0.910352,0.225274,0.227757



=== pitcher rates, season 2025  (IN SAMPLE)


...


The per-swing version was measured too. Scaling each swing about its own
player's season mean by k, the k that puts the pitcher slope at 1 is
  x 0.45  y 0.75  z 0.90
and it is NOT applied. It moves every player's rate level far more than it
moves the spread between players: pitcher category error goes from 0.021 to
0.079 on the 2024 holdout, whiff centered from 0.404 to 0.699 against
Savant's 0.402, and the pooled x histogram shifts by up to 0.186 in a single
bin. calibrate.apply_reliability exists and is tested; nothing calls it.

=== per-player Wasserstein-1 against Savant's histograms


,season,type,axis,mean_w1,median_w1,players
0,2024,pitcher,x,0.563,0.482,755
1,2024,pitcher,y,0.790,0.658,755
2,2024,pitcher,z,0.238,0.211,754
3,2024,batter,x,0.912,0.797,604
4,2024,batter,y,1.769,1.411,604
5,2024,batter,z,0.342,0.311,600
6,2026,pitcher,x,0.570,0.496,752
7,2026,pitcher,y,0.799,0.675,752
8,2026,pitcher,z,0.239,0.210,745
9,2026,batter,x,0.925,0.797,607



=== league bins against TIMING_BINNED_DATA_LEAGUE, 2025


,bin,n_ours,rv_ours,axis,n_sav,rv_sav
26,-15.0,203.0,-0.065704,x,244.0,-0.109976
27,-13.0,480.0,-0.067431,x,491.0,-0.098598
28,-11.0,1522.0,-0.112344,x,1270.0,-0.091764
29,-9.0,3477.0,-0.098875,x,3738.0,-0.084998
30,-7.0,9827.0,-0.078960,x,10734.0,-0.060335
31,-5.0,25965.0,-0.072855,x,26747.0,-0.030098
32,-3.0,51319.0,-0.046694,x,49724.0,0.004872
33,-1.0,66981.0,0.082129,x,65073.0,0.020635
34,1.0,61751.0,0.009355,x,59965.0,-0.007951
35,3.0,42668.0,-0.076708,x,42531.0,-0.051878



inversion rows past the curve's last published point: 12216 (5.35% of them). Largest single x value shared by more than one swing: 6055 swings at -0.000 inches, against 12,221 stacked on exactly 14 inches before the change.


In [7]:
# 6. The tails themselves: our value against Savant's, on the labeled whiffs.
tf = traits.tails_features(FIT, Y_COM, TILT)
# All three panels draw. Filtering to status == "recovered" left x and z blank,
# which read as a broken figure rather than as the deliberate distinction it was.
# The distinction is real and belongs in the TITLE: y is a recovered formula, and
# x and z are the best linear candidate against the tails, not what ships for
# contact swings, which come from the collision inversion instead.
tl = pd.DataFrame({ax: discover.apply(
    discover.CandidateResult(**{k: v for k, v in disc["verdicts"][ax]["best"].items()}), tf)
    for ax in ("x", "y", "z")})
status = {ax: disc["verdicts"][ax]["status"] for ax in ("x", "y", "z")}
caveat = ("Savant's tails are the ten worst whiffs per list per player-season, selected on "
          "the label. This is the population the formulas were read off, and it is not a "
          "random sample of swings: agreement here is a form check, not a generalization check.")
pred = {ax: (tl[ax] if ax in tl else np.full(len(tf), np.nan)) for ax in ("x", "y", "z")}
truth = {ax: tf[f"sav_{ax}"].to_numpy() for ax in ("x", "y", "z")}
p = plots.tails_scatter(pred, truth, FIGURES / "tails_scatter_2025.png", caveat,
                        status=status)
print(p)
display(Image(filename=str(p)))
p = plots.tails_scatter(pred, truth, FIGURES / "tails_scatter_y_2025.png", caveat,
                        axes=("y",), status=status)
print(p)
p = plots.tails_scatter(pred, truth, FIGURES / "tails_scatter_xz_2025.png", caveat,
                        axes=("x", "z"), status=status)
print(caveat)
display(Image(filename=str(p)))


data/figures/tails_scatter_2025.png


data/figures/tails_scatter_y_2025.png


Savant's tails are the ten worst whiffs per list per player-season, selected on the label. This is the population the formulas were read off, and it is not a random sample of swings: agreement here is a form check, not a generalization check.


In [8]:
sc = validate.scorecard(results, fit_target=(FIT, "pitcher by contact type"))
display(sc)
print("calibration_target marks the one table the per-type map was fit to: its band")
print("masses ARE the 2025 pitcher split CSV's rates. It is the target, not evidence.")
print("The 2025 pitcher rates row is in sample but still a real comparison, because")
print("nothing was fit to the plain CSV. 2024 and 2026 are holdouts throughout.")
sc.to_csv(REPORTS / "scorecard.csv", index=False)
header = {
    "swing filter": RULE,
    "swing filter median abs rel error": "pitcher %.4f, batter %.4f" % (
        float(rep.loc[rep.rule.eq(RULE) & rep.type.eq("pitcher"), "median_abs_rel_err"].iloc[0]),
        float(rep.loc[rep.rule.eq(RULE) & rep.type.eq("batter"), "median_abs_rel_err"].iloc[0])),
    "tail join rate": "%.5f (%d of %d, %d unmatched)" % (
        disc["join_rate"], disc["join"]["exact"] + disc["join"]["nearest"],
        disc["join"]["tails"], disc["join"]["unmatched"]),
    "discovery": "  ".join(
        f"{ax}={disc['verdicts'][ax]['status']}(r2 {disc['verdicts'][ax]['best']['r2']:.3f})"
        for ax in ("x", "y", "z")),
    "2026 snapshot: last game date": LAST_GAME.get(2026, "2026 not loaded"),
    "seasons": "fit %d, holdouts %s" % (FIT, HOLDOUT or "none yet"),
}
json.dump(header, open(REPORTS / "scorecard_header.json", "w"), indent=1)
for k, v in header.items():
    print(f"{k}: {v}")


,season,table,in_sample,calibration_target,mean_mae_rates,mean_r_rates,players
0,2024,batter by contact type,False,False,0.0428,0.503,487
1,2024,batter rates,False,False,0.0369,0.662,512
2,2024,pitcher by contact type,False,False,0.0267,0.695,560
3,2024,pitcher rates,False,False,0.0203,0.876,598
4,2025,batter by contact type,True,False,0.0412,0.508,490
5,2025,batter rates,True,False,0.0359,0.677,523
6,2025,pitcher by contact type,True,True,0.0259,0.693,570
7,2025,pitcher rates,True,False,0.0205,0.861,618
8,2026,batter by contact type,False,False,0.0436,0.507,484
9,2026,batter rates,False,False,0.0369,0.663,512


calibration_target marks the one table the per-type map was fit to: its band
masses ARE the 2025 pitcher split CSV's rates. It is the target, not evidence.
The 2025 pitcher rates row is in sample but still a real comparison, because
nothing was fit to the plain CSV. 2024 and 2026 are holdouts throughout.
swing filter: bat_speed_nobunt
swing filter median abs rel error: pitcher 0.0021, batter 0.0048
tail join rate: 0.99995 (21328 of 21329, 1 unmatched)
discovery: x=fallback(r2 0.712)  y=recovered(r2 0.951)  z=fallback(r2 0.665)
2026 snapshot: last game date: 2026-09-11
seasons: fit 2025, holdouts (2024, 2026)


## 5. Run value over the contact point and the timing

Per-swing run value from the batter's side, over every swing in every season, on
Savant's own bin grid (2 inches, 2 ms, 1 inch), with cells under 50 swings left
blank and the category thresholds drawn. The secondary view is expected wOBA on
contact, which only balls in play carry.


In [9]:
all_cal = pd.concat([CAL[s] for s in SEASONS], ignore_index=True)
all_df = pd.concat([DF[s][["delta_run_exp", "contact_type", "estimated_woba_using_speedangle",
                           "pitcher", "batter"]] for s in SEASONS], ignore_index=True)
figs = []
figs.append(plots.hexbin_grid(
    all_cal, all_df, "delta_run_exp", FIGURES / "hexbin_runvalue_all_swings.png",
    f"Run value per swing (batter perspective), all swings, {min(SEASONS)} to {max(SEASONS)}"))
ip = (all_df.contact_type == "in_play").to_numpy()
ip_cal = all_cal[ip].reset_index(drop=True)
ip_df = all_df[ip].reset_index(drop=True)
# Every panel and every figure draws xwOBA on the same fixed 0 to 1.2 scale,
# white at 0.600, and run value on -0.3 to 0.3, white at zero. plots.SCALES
# holds both; the call sites do not choose.
figs.append(plots.hexbin_grid(
    ip_cal, ip_df, "estimated_woba_using_speedangle", FIGURES / "hexbin_xwoba_in_play.png",
    f"Expected wOBA on contact, balls in play, {min(SEASONS)} to {max(SEASONS)}",
    min_n=50))

# The three axes at once. The 2D panels each marginalise one axis away; these do
# not, which is the only way to see that the best cells sit on a surface rather
# than at a point.
figs.append(plots.cloud3d(
    ip_cal, ip_df, FIGURES / "cloud3d_xwoba_in_play.png",
    f"Expected wOBA on contact over x, y and z, balls in play, "
    f"{min(SEASONS)} to {max(SEASONS)}",
    value="estimated_woba_using_speedangle", min_n=25))
figs.append(plots.cloud3d(
    all_cal, all_df, FIGURES / "cloud3d_runvalue_all_swings.png",
    f"Run value per swing over x, y and z, all swings, {min(SEASONS)} to {max(SEASONS)}",
    value="delta_run_exp", min_n=60))
CLOUD_HTML = plots.cloud3d_html(
    ip_cal, ip_df, FIGURES / "cloud3d_xwoba_in_play.html",
    f"Expected wOBA on contact over x, y and z, balls in play, "
    f"{min(SEASONS)} to {max(SEASONS)}",
    value="estimated_woba_using_speedangle", min_n=25)
print("interactive:", CLOUD_HTML)

figs.append(plots.league_bins_figure(BINS_TABLE, FIGURES / "league_bins_2025.png"))
RATES = ("on_time_percent", "centered_percent", "lined_up_percent")
for kind in ("pitcher", "batter"):
    ours = validate.player_rates(DF[FIT], CATS[FIT], kind, CAL[FIT])
    sav = validate.savant_rates(FIT, kind)
    for rate in RATES:
        figs.append(plots.rate_scatter(ours, sav, rate,
                                       FIGURES / f"scatter_{kind}_{rate}.png"))
    # All three on one figure: a single rate invites the reader to assume the
    # other two look like it, and they sit at different correlations.
    figs.append(plots.rate_scatter_grid(
        ours, sav, RATES, FIGURES / f"scatter_grid_{kind}.png",
        f"Per-{kind} category rates against Savant's leaderboard, {FIT}"))
for p in figs:
    print(p)
    display(Image(filename=str(p)))


interactive: data/figures/cloud3d_xwoba_in_play.html


data/figures/hexbin_runvalue_all_swings.png


data/figures/hexbin_xwoba_in_play.png


data/figures/cloud3d_xwoba_in_play.png


data/figures/cloud3d_runvalue_all_swings.png


...

data/figures/scatter_grid_pitcher.png


data/figures/scatter_batter_on_time_percent.png


data/figures/scatter_batter_centered_percent.png


data/figures/scatter_batter_lined_up_percent.png


data/figures/scatter_grid_batter.png
